# Phase 3 — Screener Workflow

**Purpose:** Demonstrate the screener → watchlist → snapshot → diff pipeline.
Uses the synthetic market fixture; no live data required.

In [1]:
# Cell 1: Imports and setup
import pathlib
import sys
import tempfile
from datetime import date
from pathlib import Path

import ah_research
from ah_research.analysis import ScreenResult, run_screen
from ah_research.watchlist import WatchlistStore

sys.path.insert(0, str(pathlib.Path().resolve() / "tests"))
from fixtures.phase2.synthetic_market import build_synthetic_market

code_version = getattr(ah_research, "__version__", "dev")
print(f"Python : {sys.version.split()[0]}")
print(f"ah-research: {code_version}")

Python : 3.12.10
ah-research: 0.0.1


In [2]:
# Cell 2: Build synthetic market
START = date(2023, 1, 1)
END = date(2024, 12, 31)
SYMBOLS = ["600000.SH", "000001.SZ", "600519.SH", "600036.SH"]

repo = build_synthetic_market(start=START, end=END, symbols=SYMBOLS)
print(f"Market: {len(SYMBOLS)} symbols from {START} to {END}")

Market: 4 symbols from 2023-01-01 to 2024-12-31


In [3]:
# Cell 3: Screen with loose P/E filter (all symbols should pass)
ASOF = date(2024, 12, 31)

result: ScreenResult = run_screen(
    conditions={"pe": (">", 0.0)},
    repo=repo,
    asof=ASOF,
    universe="CSI300",
)

print(f"Screen asof   : {result.asof}")
print(f"Universe      : {result.universe}")
print(f"Input symbols : {result.n_input}")
print(f"Passed filter : {result.n_passed}")
print()
if not result.frame.empty:
    display_cols = [
        c
        for c in ["symbol", "pe", "pb", "dividend_yield", "sector_l1"]
        if c in result.frame.columns
    ]
    print(result.frame[display_cols].to_string(index=False))

Screen asof   : 2024-12-31
Universe      : CSI300
Input symbols : 72
Passed filter : 72

   symbol   pe  pb  dividend_yield   sector_l1
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
600000.SH 25.0 8.0            0.02  Financials
000001.SZ 25.0 8.0

In [4]:
# Cell 4: Create watchlist from screener results
tmpdir = Path(tempfile.mkdtemp())
store = WatchlistStore(cache_path=tmpdir / "cache.duckdb")

# Use symbols from the screen result
if not result.frame.empty and "symbol" in result.frame.columns:
    screened_symbols = result.frame["symbol"].tolist()
else:
    # Fallback to all symbols if screen returned empty (synthetic data may have no PE)
    screened_symbols = SYMBOLS

wl = store.create(
    "value_screen_2024",
    symbols=screened_symbols,
    description="PE-filtered screener results as of 2024-12-31",
)

print(f"Created watchlist : {wl.name!r}")
print(f"Symbols ({len(wl.symbols)}): {[str(s) for s in wl.symbols]}")
print(f"Description       : {wl.description}")

Created watchlist : 'value_screen_2024'
Symbols (72): ['600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '600000.SH', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '000001.SZ', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600519.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH', '600036.SH']
Descripti

In [5]:
# Cell 5: Take two snapshots and compute diff
MID = date(2024, 6, 30)
LATE = date(2024, 12, 31)

snap_mid = store.snapshot("value_screen_2024", repo, asof=MID)
snap_late = store.snapshot("value_screen_2024", repo, asof=LATE)

print(f"Snapshot 1 ({MID}): {len(snap_mid.rows)} symbols")
print(f"Snapshot 2 ({LATE}): {len(snap_late.rows)} symbols")
print()
print("=== Mid-year snapshot ===")
print(snap_mid.rows.to_string(index=False))

Snapshot 1 (2024-06-30): 4 symbols
Snapshot 2 (2024-12-31): 4 symbols

=== Mid-year snapshot ===
  pe  pb  dividend_yield  roe   market_cap   sector_l1    symbol
25.0 8.0            0.02 0.25 2.000000e+12  Financials 600000.SH
25.0 8.0            0.02 0.25 2.000000e+12    Consumer 000001.SZ
25.0 8.0            0.02 0.25 2.000000e+12  Technology 600519.SH
25.0 8.0            0.02 0.25 2.000000e+12 Industrials 600036.SH


In [6]:
# Cell 6: Diff snapshots
diff = store.diff_snapshots("value_screen_2024", earlier=MID, later=LATE)

print("=== Snapshot Diff (mid → year-end) ===")
print(diff.to_string(index=False))

=== Snapshot Diff (mid → year-end) ===
   symbol  pe_delta  pb_delta  dividend_yield_delta  roe_delta  market_cap_delta
600000.SH       0.0       0.0                   0.0        0.0               0.0
000001.SZ       0.0       0.0                   0.0        0.0               0.0
600519.SH       0.0       0.0                   0.0        0.0               0.0
600036.SH       0.0       0.0                   0.0        0.0               0.0


In [7]:
# Cell 7: YAML export / import round-trip
yaml_path = tmpdir / "value_screen_2024.yaml"
store.export_yaml("value_screen_2024", yaml_path)
print(f"Exported to: {yaml_path}")
print(yaml_path.read_text())

Exported to: /var/folders/1w/s14mdy7s1s57q_ftj0hglmjm0000gn/T/tmp8oqgta8a/value_screen_2024.yaml
description: PE-filtered screener results as of 2024-12-31
name: value_screen_2024
screen_conditions: null
symbols:
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 600000.SH
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 000001.SZ
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600519.SH
- 600036.SH
- 600036.SH
- 600036.SH
- 600036.SH
- 600036.SH
- 600036.SH
- 600036.SH
- 600036.SH
- 600036.SH
- 600036.SH
- 600036.SH
- 60003

In [8]:
# Cell 8: Reproducibility block
import subprocess

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    git_sha = "unknown"

print("=== Reproducibility ===")
print(f"Code version : {code_version}")
print(f"Git SHA      : {git_sha}")
print(f"Market range : {START} to {END}")
print(f"Screen asof  : {ASOF}")
print("Data source  : synthetic (build_synthetic_market)")

=== Reproducibility ===
Code version : 0.0.1
Git SHA      : d786f88
Market range : 2023-01-01 to 2024-12-31
Screen asof  : 2024-12-31
Data source  : synthetic (build_synthetic_market)


---
> **DISCLAIMER:** Results are based on synthetic / randomly generated data. This is a workflow demonstration only and does not constitute investment advice. Past performance does not guarantee future results.